# Baby Step 2 — Tax Planning Exo-Brain
## Business-model, objective, constraint, and fragility testing

This notebook preserves Steps 0–1, creates a Step 2 copy, and stress-tests all ten synthetic Recommendation V1 records. It is an educational architecture test—not tax or legal advice.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Test architecture

Each recommendation faces eight controlled scenarios: two business-model tests, two strategic-objective tests, two binding-constraint tests, and two fragility shocks. Pass, conditional, and fail outcomes roll into a governed resilience class and next-step disposition.


In [ ]:
import csv
import hashlib
import json
import re
import shutil
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path


STEP2_QUARTER = "2026-Q3"
STEP2_TEST_VERSION = "2026-Q3-S001"
STEP2_DATE = date(2026, 7, 21).isoformat()


def read_csv(path: Path) -> list[dict]:
    with path.open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))


def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        raise ValueError(f"No rows supplied for {path}")
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.rstrip() + "\n", encoding="utf-8")


def fnum(value) -> float:
    return float(value)


SCENARIOS = [
    {
        "scenario_id": "SCN-BM-IP",
        "family": "BUSINESS_MODEL",
        "name": "IP-intensive value creation",
        "question": "Can the design support R&D, DEMPE control, and mobile-intangible scrutiny?",
        "severity": "HIGH",
    },
    {
        "scenario_id": "SCN-BM-ASSET",
        "family": "BUSINESS_MODEL",
        "name": "Asset-heavy operating model",
        "question": "Can the design support fixed assets, local execution, and operational-risk control?",
        "severity": "MEDIUM",
    },
    {
        "scenario_id": "SCN-OBJ-REPAT",
        "family": "OBJECTIVE",
        "name": "Cash repatriation priority",
        "question": "Does the structure remain attractive when dividend and interest leakage dominates?",
        "severity": "MEDIUM",
    },
    {
        "scenario_id": "SCN-OBJ-SCALE",
        "family": "OBJECTIVE",
        "name": "Rapid multi-market expansion",
        "question": "Can the hub scale across markets without outrunning treaty access and substance?",
        "severity": "MEDIUM",
    },
    {
        "scenario_id": "SCN-CON-SUB",
        "family": "CONSTRAINT",
        "name": "Substance capacity reduced",
        "question": "What happens if people, premises, or decision capacity fall by forty percent?",
        "severity": "SEVERE",
    },
    {
        "scenario_id": "SCN-CON-DOC",
        "family": "CONSTRAINT",
        "name": "Documentation deterioration",
        "question": "What happens when transfer-pricing and transaction evidence become incomplete?",
        "severity": "HIGH",
    },
    {
        "scenario_id": "SCN-FRG-TREATY",
        "family": "FRAGILITY",
        "name": "Treaty-access and withholding shock",
        "question": "What happens if entitlement is challenged and synthetic withholding rises?",
        "severity": "SEVERE",
    },
    {
        "scenario_id": "SCN-FRG-GMT",
        "family": "FRAGILITY",
        "name": "Anti-avoidance and minimum-tax shock",
        "question": "What happens under a stricter CFC and global-minimum-tax interpretation?",
        "severity": "SEVERE",
    },
]


def scenario_adjustment(scenario_id: str, industry: str, pattern: str, candidate: dict) -> tuple[float, str]:
    """Return a deterministic stress adjustment and a concise explanation."""
    industry_l = industry.lower()
    pattern_l = pattern.lower()
    substance = fnum(candidate["substance_score"])
    documentation = fnum(candidate["documentation_score"])
    anti = fnum(candidate["anti_avoidance_stability_score"])
    cit = fnum(candidate["headline_cit_rate_pct"])
    wht = fnum(candidate["synthetic_wht_parameter"])
    treaties = fnum(candidate["treaty_partner_count"])

    if scenario_id == "SCN-BM-IP":
        aligned = any(k in industry_l for k in ("digital", "software", "media", "life sciences"))
        design_fit = "ip" in pattern_l or "r&d" in pattern_l
        adjustment = (5 if aligned and design_fit else -14 if aligned else -8) + (substance - 70) * 0.08
        reason = "IP/R&D pattern and existing substance are aligned." if design_fit else "The selected pattern does not expressly anchor mobile-intangible control."
    elif scenario_id == "SCN-BM-ASSET":
        aligned = any(k in industry_l for k in ("manufacturing", "infrastructure", "energy", "logistics", "consumer"))
        design_fit = "operating" in pattern_l or "distribution" in pattern_l
        adjustment = (4 if aligned and design_fit else -12 if aligned else -6) + (substance - 65) * 0.06
        reason = "Operating pattern and physical substance support an asset-heavy model." if design_fit else "The pattern requires stronger local asset and operational-risk alignment."
    elif scenario_id == "SCN-OBJ-REPAT":
        adjustment = 6 - 0.72 * wht + 0.35 * treaties
        reason = "Synthetic withholding leakage and treaty reach drive the repatriation result."
    elif scenario_id == "SCN-OBJ-SCALE":
        adjustment = -10 + 1.35 * treaties + 0.045 * substance
        reason = "Treaty breadth and demonstrated substance determine scaling capacity."
    elif scenario_id == "SCN-CON-SUB":
        adjustment = -12 - 0.18 * substance
        reason = "A forty-percent capacity reduction weakens functional control and treaty entitlement."
    elif scenario_id == "SCN-CON-DOC":
        adjustment = -10 - 0.20 * documentation
        reason = "Incomplete evidence limits transfer-pricing defensibility and narrows permission."
    elif scenario_id == "SCN-FRG-TREATY":
        adjustment = -8 - 0.62 * wht - max(0, 7 - treaties) * 1.4
        reason = "Treaty challenge increases withholding leakage and beneficial-ownership exposure."
    elif scenario_id == "SCN-FRG-GMT":
        low_rate_exposure = max(0.0, 24.0 - cit) * 0.75
        adjustment = -9 - low_rate_exposure - (100 - anti) * 0.14
        reason = "Low-rate exposure and anti-avoidance instability create top-up and CFC fragility."
    else:
        raise KeyError(scenario_id)
    return round(adjustment, 2), reason


def apply_step2(source_vault: Path, output_vault: Path) -> dict:
    """Copy the validated Step 1 state and add governed stress and fragility testing."""
    source_vault = Path(source_vault)
    output_vault = Path(output_vault)
    required = [
        source_vault / "13_Audit" / "STEP_1_SUCCESS.md",
        source_vault / "16_Data" / "recommendations.csv",
        source_vault / "16_Data" / "recommendation_candidate_scores.csv",
        source_vault / "16_Data" / "decision_gates.csv",
        source_vault / "16_Data" / "tax_codes.csv",
        source_vault / "16_Data" / "conglomerates.csv",
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Step 1 source is incomplete:\n" + "\n".join(missing))

    if output_vault.exists():
        shutil.rmtree(output_vault)
    shutil.copytree(source_vault, output_vault)

    data = output_vault / "16_Data"
    groups = read_csv(data / "conglomerates.csv")
    recommendations = read_csv(data / "recommendations.csv")
    candidates = read_csv(data / "recommendation_candidate_scores.csv")
    decisions = read_csv(data / "decision_gates.csv")
    tax_codes = read_csv(data / "tax_codes.csv")
    entities = read_csv(data / "entities.csv")
    transactions = read_csv(data / "intercompany_transactions.csv")

    if len(groups) != 10 or len(recommendations) != 10 or len(tax_codes) != 100:
        raise AssertionError("Step 2 requires the validated 10-recommendation, 100-code Step 1 state.")

    group_by_id = {row["conglomerate_id"]: row for row in groups}
    selected_by_group = {
        row["conglomerate_id"]: row for row in candidates if row["selected"] == "TRUE"
    }
    results: list[dict] = []
    resilience_rows: list[dict] = []

    for recommendation in recommendations:
        gid = recommendation["conglomerate_id"]
        group = group_by_id[gid]
        candidate = selected_by_group[gid]
        baseline = fnum(recommendation["composite_score"])
        group_results = []

        for scenario in SCENARIOS:
            adjustment, reason = scenario_adjustment(
                scenario["scenario_id"], group["industry"], recommendation["design_pattern"], candidate
            )
            stressed = round(max(0.0, min(100.0, baseline + adjustment)), 2)
            if stressed >= 55 and adjustment >= -18:
                outcome = "PASS"
                permission = "RETAIN_FOR_REVIEW"
            elif stressed >= 40 and adjustment >= -32:
                outcome = "CONDITIONAL"
                permission = "REQUIRE_MITIGATION"
            else:
                outcome = "FAIL"
                permission = "REOPEN_RECOMMENDATION"
            row = {
                "test_id": f"TST-{gid.split('-')[1]}-{scenario['scenario_id'].replace('SCN-', '')}",
                "recommendation_id": recommendation["recommendation_id"],
                "conglomerate_id": gid,
                "scenario_id": scenario["scenario_id"],
                "family": scenario["family"],
                "scenario": scenario["name"],
                "severity": scenario["severity"],
                "baseline_score": round(baseline, 2),
                "score_adjustment": adjustment,
                "stressed_score": stressed,
                "outcome": outcome,
                "permission": permission,
                "reason": reason,
                "test_version": STEP2_TEST_VERSION,
                "synthetic": True,
            }
            results.append(row)
            group_results.append(row)

        counts = Counter(row["outcome"] for row in group_results)
        worst = min(group_results, key=lambda row: (fnum(row["stressed_score"]), row["scenario_id"]))
        average = round(sum(fnum(row["stressed_score"]) for row in group_results) / len(group_results), 2)
        downside = round(baseline - average, 2)
        if counts["FAIL"] == 0 and counts["PASS"] >= 5 and fnum(worst["stressed_score"]) >= 42:
            resilience = "ROBUST"
            disposition = "QUALIFY_FOR_STEP_3"
        elif counts["FAIL"] <= 2 and fnum(worst["stressed_score"]) >= 32:
            resilience = "MANAGED_FRAGILITY"
            disposition = "QUALIFY_WITH_MITIGATIONS"
        else:
            resilience = "FRAGILE"
            disposition = "REOPEN_BEFORE_STEP_3"

        resilience_row = {
            "recommendation_id": recommendation["recommendation_id"],
            "conglomerate_id": gid,
            "conglomerate": recommendation["conglomerate"],
            "baseline_score": round(baseline, 2),
            "average_stressed_score": average,
            "downside_from_baseline": downside,
            "worst_scenario_id": worst["scenario_id"],
            "worst_scenario": worst["scenario"],
            "worst_score": worst["stressed_score"],
            "pass_count": counts["PASS"],
            "conditional_count": counts["CONDITIONAL"],
            "fail_count": counts["FAIL"],
            "resilience_class": resilience,
            "step2_disposition": disposition,
            "test_version": STEP2_TEST_VERSION,
            "synthetic": True,
        }
        resilience_rows.append(resilience_row)

        result_lines = "\n".join(
            f"| {row['family']} | {row['scenario']} | {row['baseline_score']} | {row['score_adjustment']} | {row['stressed_score']} | {row['outcome']} |"
            for row in group_results
        )
        write_text(output_vault / "15_Application" / "Step_2_Stress_Tests" / f"STRESS-{gid}.md", f"""
---
object_type: stress_test_portfolio
conglomerate_id: {gid}
recommendation_id: {recommendation['recommendation_id']}
test_version: {STEP2_TEST_VERSION}
resilience_class: {resilience}
step2_disposition: {disposition}
synthetic: true
---

# Step 2 Stress Test — {recommendation['conglomerate']}

> Synthetic internal analysis only. Stress-test qualification does not authorize implementation or convert the recommendation into tax or legal advice.

## Recommendation tested

- Recommendation: [[{recommendation['recommendation_id']}]]
- Candidate hub: {recommendation['selected_hub_jurisdiction']}
- Design pattern: {recommendation['design_pattern']}
- Baseline score: {baseline:.2f}

## Scenario results

| Family | Scenario | Baseline | Adjustment | Stressed | Outcome |
|---|---|---:|---:|---:|---|
{result_lines}

## Resilience conclusion

- Classification: **{resilience}**
- Disposition: **{disposition}**
- Average stressed score: {average}
- Worst case: {worst['scenario']} ({worst['stressed_score']})
- Outcomes: {counts['PASS']} pass, {counts['CONDITIONAL']} conditional, {counts['FAIL']} fail.

## Governance consequence

The result controls only the next synthetic analytical step. Any conditional or failed scenario requires explicit mitigation or reopening. No real filing, transaction, restructuring, external communication, or implementation is permitted.
""")

        rec_path = output_vault / "09_Recommendations" / f"{recommendation['recommendation_id']}.md"
        rec_text = rec_path.read_text(encoding="utf-8")
        rec_text += f"\n## Step 2 stress-test result\n\n- [[STRESS-{gid}]]\n- Resilience: **{resilience}**\n- Disposition: **{disposition}**\n- Worst case: {worst['scenario']} ({worst['stressed_score']})\n"
        write_text(rec_path, rec_text)

        recommendation["step2_test_version"] = STEP2_TEST_VERSION
        recommendation["step2_resilience_class"] = resilience
        recommendation["step2_disposition"] = disposition
        recommendation["status"] = "STRESS_TESTED_PENDING_HUMAN_REVIEW"

    resilience_by_rec = {row["recommendation_id"]: row for row in resilience_rows}
    for decision in decisions:
        r = resilience_by_rec[decision["recommendation_id"]]
        decision["status"] = "STEP_2_TESTED_PENDING_HUMAN_REVIEW"
        decision["permitted_action"] = r["step2_disposition"]
        decision["implementation_authorized"] = False
        decision_path = output_vault / "10_Decisions" / f"{decision['decision_id']}.md"
        decision_text = decision_path.read_text(encoding="utf-8")
        decision_text += f"\n## Step 2 evidence\n\n- [[STRESS-{decision['conglomerate_id']}]]\n- Resilience: **{r['resilience_class']}**\n- Permitted analytical disposition: **{r['step2_disposition']}**\n- Implementation remains unauthorized.\n"
        write_text(decision_path, decision_text)

    for group in groups:
        group["recommendation_status"] = "STRESS_TESTED_PENDING_HUMAN_REVIEW"

    write_csv(data / "stress_scenarios.csv", [{**s, "test_version": STEP2_TEST_VERSION, "synthetic": True} for s in SCENARIOS])
    write_csv(data / "stress_test_results.csv", results)
    write_csv(data / "recommendation_resilience.csv", resilience_rows)
    write_csv(data / "recommendations.csv", recommendations)
    write_csv(data / "decision_gates.csv", decisions)
    write_csv(data / "conglomerates.csv", groups)

    register_lines = "\n".join(
        f"| [[{row['recommendation_id']}]] | {row['conglomerate']} | {row['baseline_score']} | {row['average_stressed_score']} | {row['worst_scenario']} | {row['worst_score']} | {row['resilience_class']} | {row['step2_disposition']} |"
        for row in resilience_rows
    )
    portfolio_counts = Counter(row["resilience_class"] for row in resilience_rows)
    outcome_counts = Counter(row["outcome"] for row in results)
    write_text(output_vault / "12_Reports" / "STEP_2_RESILIENCE_REGISTER.md", f"""
---
object_type: portfolio_report
report_id: RPT-STEP-2-001
quarter: {STEP2_QUARTER}
test_version: {STEP2_TEST_VERSION}
status: INTERNAL_SYNTHETIC_DRAFT
synthetic: true
---

# Step 2 Recommendation Resilience Register

Eight controlled scenarios were applied to each Recommendation V1: two business-model tests, two strategic-objective tests, two binding-constraint tests, and two fragility shocks.

| Recommendation | Conglomerate | Baseline | Average stressed | Worst scenario | Worst score | Resilience | Disposition |
|---|---|---:|---:|---|---:|---|---|
{register_lines}

## Portfolio result

- {len(results)} total scenario evaluations.
- Outcomes: {outcome_counts['PASS']} pass, {outcome_counts['CONDITIONAL']} conditional, {outcome_counts['FAIL']} fail.
- Resilience: {portfolio_counts['ROBUST']} robust, {portfolio_counts['MANAGED_FRAGILITY']} managed fragility, {portfolio_counts['FRAGILE']} fragile.
- A high baseline score does not override a failed stress scenario.
- Every recommendation remains subject to human review and internal-scenario-only permission.
""")

    write_text(output_vault / "11_Quarterly_Updates" / STEP2_QUARTER / "STEP_2_STRESS_TEST_CYCLE.md", f"""
# {STEP2_QUARTER} — Step 2 Stress-Test Cycle

- Recommendation V1 records tested: 10.
- Scenario families: business model, objective, constraint, and fragility.
- Standardized scenarios per recommendation: 8.
- Total evaluations: {len(results)}.
- Tax-code alterations this cycle: 0.
- New companies this cycle: 0.
- Implementation authority: none.
""")

    write_text(output_vault / "14_Hot_Cache" / "CURRENT_STATE.md", f"""
# Current State — {STEP2_QUARTER} Stress-Tested Recommendation V1

- Active step: 2 of 10
- Tax-code modules: 100 (unchanged)
- Conglomerates: 10 (unchanged)
- Recommendations stress-tested: 10
- Scenario evaluations: {len(results)}
- Test version: {STEP2_TEST_VERSION}
- Robust: {portfolio_counts['ROBUST']}
- Managed fragility: {portfolio_counts['MANAGED_FRAGILITY']}
- Fragile/reopen: {portfolio_counts['FRAGILE']}
- Next analytical layer: Step 3, formal provenance and atomic claims
- Explicit prohibition: no real tax advice, filing, transaction, communication, or restructuring
""")

    state = {
        "project": "Tax Planning Exo-Brain",
        "active_step": 2,
        "quarter": STEP2_QUARTER,
        "recommendation_version": "2026-Q3-R001",
        "stress_test_version": STEP2_TEST_VERSION,
        "counts": {
            "jurisdictions": 20,
            "tax_codes": len(tax_codes),
            "conglomerates": len(groups),
            "entities": len(entities),
            "transactions": len(transactions),
            "recommendations": len(recommendations),
            "stress_scenarios": len(SCENARIOS),
            "stress_test_results": len(results),
            "robust": portfolio_counts["ROBUST"],
            "managed_fragility": portfolio_counts["MANAGED_FRAGILITY"],
            "fragile": portfolio_counts["FRAGILE"],
            "decision_gates": len(decisions),
        },
        "permission": "INTERNAL_SCENARIO_ONLY",
        "implementation_authorized": False,
        "synthetic": True,
    }
    write_text(output_vault / "00_System" / "CURRENT_STATE.json", json.dumps(state, indent=2))

    excluded = {"13_Audit/STEP_2_VALIDATION.json", "13_Audit/STEP_2_MANIFEST.csv", "13_Audit/STEP_2_SUCCESS.md"}
    manifest_rows = []
    for path in sorted(p for p in output_vault.rglob("*") if p.is_file()):
        rel = path.relative_to(output_vault).as_posix()
        if rel in excluded:
            continue
        manifest_rows.append({
            "path": rel,
            "bytes": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        })
    write_csv(output_vault / "13_Audit" / "STEP_2_MANIFEST.csv", manifest_rows)

    checks = {
        "step1_success_marker_present": (output_vault / "13_Audit" / "STEP_1_SUCCESS.md").exists(),
        "tax_codes_equal_100": len(tax_codes) == 100,
        "conglomerates_equal_10": len(groups) == 10,
        "recommendations_equal_10": len(recommendations) == 10,
        "scenarios_equal_8": len(SCENARIOS) == 8,
        "four_scenario_families": set(s["family"] for s in SCENARIOS) == {"BUSINESS_MODEL", "OBJECTIVE", "CONSTRAINT", "FRAGILITY"},
        "tests_equal_80": len(results) == 80,
        "eight_tests_per_recommendation": all(v == 8 for v in Counter(r["recommendation_id"] for r in results).values()),
        "all_test_outcomes_valid": all(r["outcome"] in {"PASS", "CONDITIONAL", "FAIL"} for r in results),
        "resilience_records_equal_10": len(resilience_rows) == 10,
        "all_resilience_classes_valid": all(r["resilience_class"] in {"ROBUST", "MANAGED_FRAGILITY", "FRAGILE"} for r in resilience_rows),
        "all_recommendations_stress_tested": all(r["status"] == "STRESS_TESTED_PENDING_HUMAN_REVIEW" for r in recommendations),
        "decision_gates_equal_10": len(decisions) == 10,
        "implementation_never_authorized": not any(str(r["implementation_authorized"]).lower() == "true" for r in decisions),
        "no_tax_code_versions_changed": all(r["version"] == "2026-Q3-V001" for r in tax_codes),
        "no_new_companies": len(groups) == 10,
        "formal_claim_layer_still_empty": not any((output_vault / "07_Atomic_Claims").glob("*.md")),
        "formal_contradiction_layer_still_empty": not any((output_vault / "08_Contradictions").glob("*.md")),
    }
    validation = {
        "step": 2,
        "date": STEP2_DATE,
        "quarter": STEP2_QUARTER,
        "test_version": STEP2_TEST_VERSION,
        "checks": checks,
        "counts": state["counts"],
        "outcomes": dict(outcome_counts),
        "resilience": dict(portfolio_counts),
        "status": "PASS" if all(checks.values()) else "FAIL",
    }
    write_text(output_vault / "13_Audit" / "STEP_2_VALIDATION.json", json.dumps(validation, indent=2))
    if validation["status"] != "PASS":
        failed = [key for key, ok in checks.items() if not ok]
        raise AssertionError("Step 2 validation failed: " + ", ".join(failed))
    write_text(output_vault / "13_Audit" / "STEP_2_SUCCESS.md", f"""
# Step 2 Validation: PASS

Validated {len(results)} scenario evaluations across four test families for ten Recommendation V1 records. Portfolio result: {portfolio_counts['ROBUST']} robust, {portfolio_counts['MANAGED_FRAGILITY']} managed fragility, and {portfolio_counts['FRAGILE']} fragile. No implementation is authorized.
""")
    return validation


In [ ]:
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/Tax_Planning_ExoBrain_Project')
SOURCE_VAULT = PROJECT / 'Step_1' / 'Tax_Planning_ExoBrain_Vault'
OUTPUT_VAULT = PROJECT / 'Step_2' / 'Tax_Planning_ExoBrain_Vault'
validation = apply_step2(SOURCE_VAULT, OUTPUT_VAULT)
print(json.dumps(validation, indent=2))
print(f'\nStep 2 vault created at: {OUTPUT_VAULT}')


## Expected result

The validation status must be `PASS`: 100 unchanged tax-code modules, 10 unchanged conglomerates, 8 scenarios, 80 stress-test results, 10 resilience records, and 10 human decision gates. No implementation is authorized.
